# 410. 수강생 실습 — RAG Agent (검색 도구를 가진 에이전트)

## 학습 목표
[320 RAG ClovaX](320_RAG_ClovaX.py) 의 검색-증강 생성을 [400 Tools & Agents](400_Tools_Agents.py)
의 에이전트 구조 위에 올려, **에이전트가 필요할 때만 검색 도구를 호출** 하는
진짜 RAG Agent 를 만들어 봅니다.

## 320 (RAG) vs 410 (RAG Agent) 의 차이
| 항목 | 320. RAG (Chain-based) | **410. RAG Agent (Agentic)** |
|------|------------------------|------------------------------|
| 검색 시점 | **항상** 먼저 검색 → 생성 | 모델이 **필요할 때만** 검색 도구 호출 |
| 구조 | 고정된 함수 호출 | LLM 의 자율적 도구 선택 |
| 장점 | 단순하고 예측 가능 | 유연함, 단순 질문엔 검색 생략 가능 |
| 단점 | 모든 질문에 검색 비용 발생 | 여러 번의 LLM 호출 → 지연 시간 증가 |
| 비유 | "무조건 책 찾아보고 답해" | "필요하면 책 찾아보고 답해" |

## RAG Agent 워크플로우
```
  [인덱싱 단계 — 1회 수행]
    문서 로드 → 청크 분할 → 임베딩 → 벡터 스토어 저장

  [질의 응답 단계 — 매번 수행]
    사용자 질문 → [LLM: 검색 필요?]
                    │
                    ├─ Yes → retrieve_context 도구 호출 → 답변 생성
                    │
                    └─ No → 바로 답변 생성
```

## 사용 기술
- **LLM**: Gemini (`gemini-2.5-flash`)
- **임베딩**: Google `models/gemini-embedding-001`
- **벡터 스토어**: LangChain `InMemoryVectorStore` (실서비스는 Chroma/Pinecone 등)
- **문서 로더**: `WebBaseLoader` (웹 페이지에서 텍스트 추출)
- **에이전트**: LangChain v1 `create_agent`

---
## 0. 라이브러리 설치
Colab 에서 실행한다면 아래 설치 명령을 먼저 실행하세요.

In [ ]:
!pip install -q -U langchain langchain-google-genai langchain-community beautifulsoup4

---
## 과제 1. 환경 설정 — 모델 / 임베딩 / 벡터 스토어 초기화

RAG Agent 는 세 가지 부품이 필요합니다:
1. **생성 모델** (Gemini): 최종 답변을 만드는 LLM
2. **임베딩 모델**: 문서/질문을 벡터로 변환
3. **벡터 스토어**: 임베딩을 저장하고 유사도 검색을 수행

**할 일**:
- `.env` 의 `GOOGLE_API_KEY` 를 로드하세요.
- `init_chat_model("gemini-2.5-flash", ...)` 로 모델을 만드세요.
- `GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")` 로 임베딩을 만드세요.
- `InMemoryVectorStore(embeddings)` 로 메모리 기반 벡터 스토어를 만드세요.

**힌트**: `InMemoryVectorStore` 는 실습용으로 충분하지만 프로세스가 끝나면 사라집니다.
실서비스에서는 Chroma / Pinecone / Weaviate 등의 **영구 vector DB** 로 바꿔야 합니다.

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()

In [ ]:
from langchain.chat_models import init_chat_model
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_core.vectorstores import InMemoryVectorStore

# 모델 및 임베딩 초기화 (Gemini 사용)
model = init_chat_model("gemini-2.5-flash", model_provider="google_genai")
embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")

# 벡터 스토어 생성
vector_store = InMemoryVectorStore(embeddings)

**관찰 포인트**
- 320 노트북에서 직접 dict + sklearn 으로 만들었던 벡터 저장소가, 여기서는
  LangChain 의 표준 `InMemoryVectorStore` 한 줄로 대체됩니다.
- 같은 인터페이스(`similarity_search`, `add_documents`) 를 따르는 한, 메모리 스토어를
  Chroma 나 Pinecone 으로 갈아 끼워도 위에 얹힌 코드는 그대로 통합니다 — LangChain
  추상화의 큰 장점입니다.

---
## 과제 2. 웹 문서 로드 및 청크 분할 (Indexing — 1/2)

실제 RAG 시스템에서는 PDF·웹페이지·DB 등 다양한 소스에서 문서를 불러옵니다.
이번엔 **공개 블로그 글 한 편** 을 인터넷에서 가져와 청크로 자릅니다.

**할 일**:
- `WebBaseLoader("https://botpress.com/ko/blog/llm-agents")` 로 페이지를 로드하세요.
- 로드된 문서 수와 본문 길이를 출력해 확인하세요.
- `RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)` 로 청크를 분할하세요.

**힌트**:
- `RecursiveCharacterTextSplitter` 는 문단·줄·문장·단어 우선순위로 점진적으로 쪼개는
  "스마트한" 분할기입니다. 320 의 단순 문자 단위 분할보다 의미 단위가 잘 유지됩니다.
- `chunk_overlap=200` 으로 청크 사이에 겹치는 부분을 두면 경계에서 문맥이 끊기지 않습니다.

In [ ]:
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 웹 문서 로드
loader = WebBaseLoader("https://botpress.com/ko/blog/llm-agents")
docs = loader.load()

print(f"로드된 문서 수: {len(docs)}")
print(f"문서 길이: {len(docs[0].page_content)} 문자")

# 텍스트 분할
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

all_splits = text_splitter.split_documents(docs)
print(f"\n분할된 청크 수: {len(all_splits)}")

**관찰 포인트**
- 청크 1개 = 약 1000 자 + 앞뒤로 200 자 overlap → 인접 청크는 약 800 자 간격으로 시작합니다.
- 청크 수가 너무 많으면 검색은 정밀하지만 LLM 호출 비용·지연이 증가합니다.
- 1000 / 200 은 한국어 일반 블로그 글에서 무난한 기본값입니다.

---
## 과제 3. 벡터 스토어에 저장 (Indexing — 2/2)

청크들을 벡터 스토어에 넣으면, 내부적으로 자동으로 **임베딩으로 변환되어** 저장됩니다.

**할 일**:
- `vector_store.add_documents(documents=all_splits)` 로 청크를 저장하세요.
- 반환된 문서 ID 수를 출력해 모두 저장됐는지 확인하세요.

**힌트**: 임베딩 API 호출이 청크 개수만큼 발생하므로 첫 실행 시 약간 시간이 걸립니다.
Gemini 임베딩은 한 번에 여러 청크를 묶어 batch 로 보내 효율을 올립니다.

In [ ]:
# 문서를 벡터 스토어에 추가
document_ids = vector_store.add_documents(documents=all_splits)
print(f"저장된 문서 ID 수: {len(document_ids)}")

---
## 과제 4. 체인 기반 RAG (Chain-based RAG) — 비교용 baseline

RAG 의 가장 단순한 형태 — **무조건 검색 → 무조건 생성**. 320 노트북과 동일한 방식입니다.
Agentic RAG (과제 5~7) 와의 비교 기준선이 됩니다.

**할 일**:
- `chain_based_rag(query)` 함수를 만들어:
  1. `vector_store.similarity_search(query, k=2)` 로 상위 2개 청크 검색
  2. 청크를 합쳐 `context` 문자열로 만들고 프롬프트에 끼워 LLM 에 전달
- "에이전트의 주요 특징은?" 으로 테스트하세요.

**힌트**: 320 에서 직접 구현했던 `rag_query` 와 본질적으로 동일합니다.
차이는 vector store 와 LLM 이 LangChain 추상화 뒤에 깔끔하게 숨겨졌다는 점뿐.

In [ ]:
# 체인 기반 RAG 예제
from langchain_core.messages import HumanMessage


def chain_based_rag(query: str):
    """체인 기반 RAG: 항상 검색 후 답변 생성"""
    # 1. 검색
    retrieved_docs = vector_store.similarity_search(query, k=2)
    context = "\n\n".join(doc.page_content for doc in retrieved_docs)

    # 2. 컨텍스트와 함께 답변 생성
    messages = [
        HumanMessage(content=f"""다음 컨텍스트를 참고하여 질문에 답변하세요:

컨텍스트:
{context}

질문: {query}

답변:""")
    ]

    response = model.invoke(messages)
    return response.content


# 체인 기반 RAG 테스트
query = "에이전트의 주요 특징은?"
print(f"질문: {query}\n")
answer = chain_based_rag(query)
print(f"답변: {answer}")

**관찰 포인트**
- 이 함수는 **모든 질문에 무조건 검색** 을 수행합니다. "안녕하세요" 같은 인사에도
  불필요하게 벡터 검색이 돌아갑니다 — 단순함의 비용.
- 다음 과제부터는 LLM 이 직접 "검색이 필요한가?" 를 결정합니다.

---
## 과제 5. 검색 도구 정의 (`retrieve_context`)

체인 방식의 `vector_store.similarity_search(...)` 호출을 **LLM 이 호출할 수 있는 도구** 로
한 번 더 감쌉니다. 그러면 에이전트가 "이 질문엔 검색이 필요해" 라고 판단했을 때만
이 도구를 호출하게 됩니다.

**할 일**:
- `@tool(response_format="content_and_artifact")` 데코레이터로 `retrieve_context(query: str)` 함수를 만드세요.
- 함수는 (1) 검색 결과 문자열과 (2) 원본 Document 객체 리스트를 **튜플로** 반환합니다.
- "LLM 에이전트의 구성 요소" 로 도구를 직접 호출해 결과를 확인하세요.

**힌트**:
- `response_format="content_and_artifact"` → 도구가 두 값을 반환할 수 있게 해줍니다.
  첫 번째는 LLM 에게 보낼 텍스트, 두 번째는 메타데이터로 남겨 둘 원본 객체.
- docstring 에 "어떤 상황에 사용해야 하는지" 를 명확히 적어 두면 모델이 더 적절히 선택합니다.

In [ ]:
from langchain.tools import tool


@tool(response_format="content_and_artifact")
def retrieve_context(query: str):
    """질문에 답하기 위해 관련 정보를 검색합니다.

    이 도구는 벡터 스토어에서 질문과 관련된 문서를 검색합니다.
    검색된 문서는 답변 생성에 사용됩니다.

    Args:
        query: 검색할 질문 또는 키워드
    """
    # 벡터 스토어에서 유사한 문서 검색
    retrieved_docs = vector_store.similarity_search(query, k=2)

    # 검색된 문서를 문자열로 직렬화
    serialized = "\n\n".join(
        f"출처: {doc.metadata.get('source', 'N/A')}\n내용: {doc.page_content}"
        for doc in retrieved_docs
    )

    # 문자열과 문서 객체를 함께 반환
    return serialized, retrieved_docs


# 도구 테스트
test_result = retrieve_context.invoke("LLM 에이전트의 구성 요소")
test_result

**관찰 포인트**
- 도구는 일반 함수처럼 `.invoke({...})` 로 직접 호출도 가능합니다 — 디버깅에 유용.
- 반환값에 `출처:` 메타데이터를 포함시켰으므로 LLM 이 답변에서 인용 출처를 함께 보여줄 수 있습니다.

---
## 과제 6. RAG 에이전트 생성 (`create_agent` + system_prompt)

도구가 준비되었으니 이제 에이전트에 묶어 "스스로 도구를 골라 호출하는 RAG" 를 만듭니다.

**할 일**:
- 적절한 `system_prompt` 를 작성하세요 — "필요한 정보를 retrieve_context 로 먼저 검색하고
  답변하라" 는 지시가 핵심입니다.
- `create_agent(model, tools=[retrieve_context], system_prompt=system_prompt)` 로 에이전트를 만드세요.

**힌트**: system_prompt 에 "먼저 검색 도구를 사용한 뒤 답하라" 라고 강하게 유도하지
않으면 모델이 검색 없이 자기 지식으로 답해 버리는 경우가 있습니다.

In [ ]:
from langchain.agents import create_agent

# 시스템 프롬프트 설정
system_prompt = (
    "당신은 블로그 게시글에서 관련 문맥(context)을 검색하는 도구에 접근할 수 있습니다. "
    "사용자의 질문에 답하기 위해 먼저 retrieve_context 도구를 사용하여 관련 정보를 검색한 후, "
    "검색된 정보를 바탕으로 정확하고 유용한 답변을 제공하세요."
)

# 에이전트 생성
agent = create_agent(
    model,
    tools=[retrieve_context],
    system_prompt=system_prompt
)

agent

---
## 과제 7. 에이전트 실행 (스트리밍 모드)

`agent.stream(...)` 로 에이전트의 **각 단계 메시지** 를 순차적으로 받아볼 수 있습니다.
(system → user → AI tool_call → tool result → AI 최종 답변) 흐름이 한 번에 보입니다.

**할 일**:
- "LLM 에이전트 프레임워크를 구성하는 핵심 구성 요소는 무엇인가요?" 질문으로
  `agent.stream(..., stream_mode="values")` 를 호출하세요.
- 매 step 의 `event["messages"][-1].pretty_print()` 로 흐름을 시각화하세요.

**힌트**:
- `stream_mode="values"` → 매 step 마다 전체 state 를 반환. 마지막 메시지만 보면 흐름 파악이 쉬움.
- 다른 모드 `"updates"` 는 변경분만 반환하므로 더 짧은 출력이 필요할 때 사용.

In [ ]:
# 질문에 대한 답변 생성
query = "LLM 에이전트 프레임워크를 구성하는 핵심 구성 요소는 무엇인가요?"
# query = "에이전트의 주요 특징은 무엇인가요?"
# query = "프롬프트 엔지니어링이 중요한 이유는 무엇인가요?"

print(f"질문: {query}\n")
print("=" * 80)

# 스트리밍 방식으로 실행
for event in agent.stream(
    {"messages": [{"role": "user", "content": query}]},
    stream_mode="values",
):
    event["messages"][-1].pretty_print()

**관찰 포인트 — 에이전트의 ReAct 루프 한 사이클**
- 첫 메시지(System) → 사용자 질문(Human) → AI 가 `retrieve_context` 호출 결정(`tool_calls`)
  → 검색 도구가 청크 반환(Tool message) → AI 가 청크를 보고 최종 답변 생성(AI).
- "안녕하세요" 같은 단순 메시지를 던지면 검색 도구가 호출되지 않을 수도 있습니다 — 단순
  질문엔 검색을 생략하는 것이 Agentic RAG 의 진짜 장점입니다.
- 같은 질문이라도 모델·프롬프트·온도 설정에 따라 결과가 다를 수 있는 비결정적 동작입니다.

---
## 종합 정리

| 단계 | Chain-based RAG (320) | **Agentic RAG (410)** |
|------|------------------------|------------------------|
| 인덱싱 | 직접 dict + sklearn | LangChain `InMemoryVectorStore` |
| 검색 시점 | 항상 자동 호출 | LLM 이 필요할 때만 도구로 호출 |
| 검색 방식 | 함수 직접 호출 | `@tool` 로 래핑된 도구 |
| 제어 | 코드가 흐름 결정 | LLM 이 흐름 결정 |
| 호출 횟수 | LLM 1회 | LLM 2회 이상 (Reason → Act → Observe → Answer) |
| 단점 | 모든 질문에 검색 비용 | 지연 시간 증가 |
| 적합한 상황 | FAQ·간단한 QA | 복잡한 의사결정, 다중 도구 통합 |

**핵심 메시지**:
- RAG Agent = "검색 도구를 가진 에이전트" 일 뿐, 마법이 아닙니다. 320 의 RAG 와 400 의
  Agent 두 개념을 합친 결과물입니다.
- **선택의 기준**: 답변이 항상 외부 지식에 의존해야 한다면 Chain-based 가 단순하고 빠릅니다.
  질문 성격이 다양해서 "검색이 필요한 것" 과 "필요 없는 것" 이 섞여 있다면 Agentic 이 효율적입니다.
- 도구를 잘 정의하는 것 (이름·설명·반환값 구조) 이 에이전트의 성능을 좌우합니다.

---
## 추가 실습 (선택 과제)

1. "오늘 점심 추천해줘" 같이 **블로그와 무관한** 질문을 던져서, 에이전트가 검색 도구를
   호출하지 않고 자기 지식으로 답하는지 관찰하세요.
2. `system_prompt` 를 "어떤 질문에도 답하기 전에 반드시 검색 도구를 호출하세요" 로 더
   강하게 바꿔, 무관한 질문에도 검색이 호출되는지 비교하세요.
3. `chain_based_rag` 와 `agent.stream` 으로 **같은 질문** 을 처리할 때의 (1) 응답 시간,
   (2) LLM 호출 횟수, (3) 답변 품질을 비교하세요.
4. `WebBaseLoader` 의 URL 을 본인이 관심 있는 한국어 블로그 글로 바꾸고, 그 글에 대해
   질문하는 전용 QA 봇을 만들어 보세요.
5. (도전) 도구 2개 — `retrieve_context` (블로그 검색) + `get_weather` (400 의 날씨 API) —
   를 함께 가진 에이전트를 만들어 "비 오는데 LLM 에이전트 학습하기 좋을까?" 같은 복합
   질문에 두 도구가 모두 호출되는지 관찰하세요.